# model_svm.py

Support Vector Machine for Heart Disease / Cardiovascular Risk prediction.

Steps:
  1. Load preprocessed data
  2. Define model architecture
  3. Define loss (hinge / built-in SVC)
  4. Define optimiser (SGD via LinearSVC or kernel SVC)
  5. Train with cross-validation (simulated epochs via batch CV)
  6. Evaluate
  7. Fine-tune (GridSearchCV)
  8. Save model
  9. Predict on new data

In [2]:
import numpy as np
import pandas as pd
import joblib
import os
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score, ConfusionMatrixDisplay)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
PRE   = "preprocessed"
MDIR  = "models/svm";  os.makedirs(MDIR, exist_ok=True)
PDIR  = "plots/svm";   os.makedirs(PDIR, exist_ok=True)
LABEL = {0: 'Low', 1: 'Moderate', 2: 'High'}

## 1. Load preprocessed splits

In [4]:
X_train = pd.read_csv(f"{PRE}/X_train.csv")
X_val   = pd.read_csv(f"{PRE}/X_val.csv")
X_test  = pd.read_csv(f"{PRE}/X_test.csv")
y_train = pd.read_csv(f"{PRE}/y_train.csv").squeeze()
y_val   = pd.read_csv(f"{PRE}/y_val.csv").squeeze()
y_test  = pd.read_csv(f"{PRE}/y_test.csv").squeeze()
class_weights = joblib.load(f"{PRE}/class_weights.pkl")

print(f"Train {X_train.shape} | Val {X_val.shape} | Test {X_test.shape}")

Train (51229, 30) | Val (10978, 30) | Test (10978, 30)


## 2. Define model architecture SGDClassifier with hinge loss ≈ linear SVM (supports partial_fit → batch/epoch training)

In [6]:
BATCH_SIZE = 256
N_EPOCHS   = 20
LR         = 0.001           # regularisation strength C = 1/alpha/n_samples

sgd_svm = SGDClassifier(
    loss             = 'hinge',    # SVM loss
    penalty          = 'l2',
    alpha            = LR,
    class_weight     = class_weights,
    random_state     = 42,
    max_iter         = 1,          # handled manually below
    warm_start       = True,
    n_jobs           = -1,
    tol              = None,       # disable auto-stop; we control epochs
)

classes = np.unique(y_train)

## 3. Loss: hinge (implicit in SGDClassifier) We track 0-1 accuracy as proxy loss per epoch

## 4. Optimiser: SGD (stochastic gradient descent) lr schedule: constant; set via alpha above

## 5. Train for N epochs in batches

In [10]:
X_tr_arr = X_train.values
y_tr_arr = y_train.values
X_vl_arr = X_val.values
y_vl_arr = y_val.values

train_acc_hist, val_acc_hist = [], []
print("\n── Training SVM (SGD, hinge) ──")
for epoch in range(1, N_EPOCHS + 1):
    idx = np.random.permutation(len(X_tr_arr))
    X_shuf, y_shuf = X_tr_arr[idx], y_tr_arr[idx]

    for start in range(0, len(X_shuf), BATCH_SIZE):
        Xb = X_shuf[start:start + BATCH_SIZE]
        yb = y_shuf[start:start + BATCH_SIZE]
        sgd_svm.partial_fit(Xb, yb, classes=classes)

    tr_acc = accuracy_score(y_tr_arr, sgd_svm.predict(X_tr_arr))
    vl_acc = accuracy_score(y_vl_arr, sgd_svm.predict(X_vl_arr))
    train_acc_hist.append(tr_acc)
    val_acc_hist.append(vl_acc)
    print(f"  Epoch {epoch:02d}/{N_EPOCHS}  train_acc={tr_acc:.4f}  val_acc={vl_acc:.4f}")


── Training SVM (SGD, hinge) ──
  Epoch 01/20  train_acc=0.9331  val_acc=0.9353
  Epoch 02/20  train_acc=0.9294  val_acc=0.9281
  Epoch 03/20  train_acc=0.9361  val_acc=0.9379
  Epoch 04/20  train_acc=0.9245  val_acc=0.9250
  Epoch 05/20  train_acc=0.9271  val_acc=0.9269
  Epoch 06/20  train_acc=0.9303  val_acc=0.9302
  Epoch 07/20  train_acc=0.9382  val_acc=0.9412
  Epoch 08/20  train_acc=0.9288  val_acc=0.9294
  Epoch 09/20  train_acc=0.9166  val_acc=0.9171
  Epoch 10/20  train_acc=0.9328  val_acc=0.9331
  Epoch 11/20  train_acc=0.9239  val_acc=0.9240
  Epoch 12/20  train_acc=0.8942  val_acc=0.8957
  Epoch 13/20  train_acc=0.9310  val_acc=0.9313
  Epoch 14/20  train_acc=0.9324  val_acc=0.9344
  Epoch 15/20  train_acc=0.9301  val_acc=0.9306
  Epoch 16/20  train_acc=0.9245  val_acc=0.9240
  Epoch 17/20  train_acc=0.9377  val_acc=0.9393
  Epoch 18/20  train_acc=0.9306  val_acc=0.9322
  Epoch 19/20  train_acc=0.9363  val_acc=0.9390
  Epoch 20/20  train_acc=0.9291  val_acc=0.9289


## 6. Evaluate on Test Set

In [12]:
print("\n── Evaluation on Test Set ──")
y_pred = sgd_svm.predict(X_test.values)
print(classification_report(y_test, y_pred, target_names=[LABEL[i] for i in sorted(LABEL)]))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Calibrate for probability output (needed for ROC-AUC)
cal_svm = CalibratedClassifierCV(sgd_svm, cv='prefit')
cal_svm.fit(X_vl_arr, y_vl_arr)
y_proba = cal_svm.predict_proba(X_test.values)
print("ROC-AUC (OvR):", roc_auc_score(y_test, y_proba, multi_class='ovr'))


── Evaluation on Test Set ──
              precision    recall  f1-score   support

         Low       0.96      0.97      0.96      7669
    Moderate       0.90      0.88      0.89      3173
        High       0.05      0.07      0.06       136

    accuracy                           0.93     10978
   macro avg       0.64      0.64      0.64     10978
weighted avg       0.93      0.93      0.93     10978

Accuracy : 0.9314993623610858
Confusion Matrix:
 [[7416  216   37]
 [ 237 2801  135]
 [  48   79    9]]
ROC-AUC (OvR): 0.9213974986596677


## 7. Fine-tune with GridSearchCV (kernel SVC)

In [14]:
print("\n── Fine-Tuning (Kernel SVC – GridSearch) ──")
param_grid = {
    'C'      : [0.1, 1, 10],
    'kernel' : ['rbf', 'poly'],
    'gamma'  : ['scale', 'auto'],
}
svc_fine = SVC(class_weight=class_weights, probability=True, random_state=42, max_iter=3000)
grid_cv  = GridSearchCV(svc_fine, param_grid, cv=StratifiedKFold(3),
                        scoring='accuracy', n_jobs=-1, verbose=1)
grid_cv.fit(X_train, y_train)
best_svc = grid_cv.best_estimator_
print("Best params:", grid_cv.best_params_)

y_pred_ft = best_svc.predict(X_test)
print("Fine-tuned Accuracy :", accuracy_score(y_test, y_pred_ft))
print(classification_report(y_test, y_pred_ft, target_names=[LABEL[i] for i in sorted(LABEL)]))


── Fine-Tuning (Kernel SVC – GridSearch) ──
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best params: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
Fine-tuned Accuracy : 0.8638185461832757
              precision    recall  f1-score   support

         Low       0.99      0.83      0.90      7669
    Moderate       0.97      0.96      0.97      3173
        High       0.03      0.25      0.05       136

    accuracy                           0.86     10978
   macro avg       0.66      0.68      0.64     10978
weighted avg       0.97      0.86      0.91     10978



## 8. Save models

In [16]:
joblib.dump(sgd_svm,  f"{MDIR}/sgd_svm_base.pkl")
joblib.dump(cal_svm,  f"{MDIR}/svm_calibrated.pkl")
joblib.dump(best_svc, f"{MDIR}/svm_finetuned.pkl")
print(f"\n✅ Models saved to ./{MDIR}/")

# Save plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_acc_hist, label='Train Acc')
axes[0].plot(val_acc_hist,   label='Val Acc')
axes[0].set(title='SVM Training Curve', xlabel='Epoch', ylabel='Accuracy')
axes[0].legend()

cm = confusion_matrix(y_test, y_pred_ft)
ConfusionMatrixDisplay(cm, display_labels=[LABEL[i] for i in sorted(LABEL)]).plot(ax=axes[1])
axes[1].set_title('SVM Fine-tuned Confusion Matrix')
plt.tight_layout()
plt.savefig(f"{PDIR}/svm_results.png", dpi=150)
print(f"Plot saved to ./{PDIR}/svm_results.png")


✅ Models saved to ./models/svm/
Plot saved to ./plots/svm/svm_results.png


## 9. Predict on new data

In [18]:
def predict_new(raw_df: pd.DataFrame, model_path: str = f"{MDIR}/svm_finetuned.pkl"):
    """
    Accepts a DataFrame of new raw samples (same columns as training features
    AFTER preprocessing – i.e. post preprocessing.py transformations).
    Returns predicted risk labels and probabilities.
    """
    scaler        = joblib.load(f"{PRE}/scaler.pkl")
    feature_names = joblib.load(f"{PRE}/feature_names.pkl")
    model         = joblib.load(model_path)

    # Align columns
    raw_df = raw_df.reindex(columns=feature_names, fill_value=0)

    numerical_cols = [
        'Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption',
        'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption',
        'Checkup_Frequency', 'Lifestyle_Score', 'Healthy_Diet_Score',
        'Smoking_Alcohol', 'Checkup_Exercise', 'Height_to_Weight',
        'Fruit_Vegetables', 'HealthyDiet_Lifestyle', 'Alcohol_FriedPotato',
        'Risk_Composite', 'Alcohol_per_Weight'
    ]
    num_present = [c for c in numerical_cols if c in raw_df.columns]
    raw_df[num_present] = scaler.transform(raw_df[num_present])

    preds  = model.predict(raw_df)
    labels = [LABEL[p] for p in preds]
    probas = model.predict_proba(raw_df) if hasattr(model, 'predict_proba') else None
    return labels, probas

# Demo call on first 5 test rows
sample = X_test.iloc[:5].copy()
# Inverse-transform is not needed; we pass already-scaled data directly
model_loaded = joblib.load(f"{MDIR}/svm_finetuned.pkl")
demo_preds = model_loaded.predict(sample.values)
print("\nDemo predictions (first 5 test samples):",
      [LABEL[p] for p in demo_preds],
      "| Actual:", list(y_test.iloc[:5].map(LABEL)))


Demo predictions (first 5 test samples): ['Low', 'Moderate', 'Low', 'Low', 'Low'] | Actual: ['Low', 'Moderate', 'Low', 'Low', 'Low']
